# FracAtlas fracture detection: ConvNeXt, DINOv2 and YOLO

Companion notebook for [MHKhan50/fracatlas-fracture-benchmark](https://github.com/MHKhan50/fracatlas-fracture-benchmark).
Image-level fracture classification on FracAtlas (4,083 musculoskeletal X-rays, 717 with fractures; CC BY 4.0) with a small CNN,
ConvNeXt-Tiny, DINOv2-S (frozen probe and fine-tuned), three seeds each, plus YOLO localisation, calibration and abstention.

**Research only. Not a medical device.**

Kaggle settings: **GPU T4**, **Internet on**. *Save Version → Save & Run All*. A smoke test runs first; the full run takes about 3 hours.
If figshare is unreachable, attach a Kaggle copy of FracAtlas (Add Input) and it is used instead.
Outputs: `/kaggle/working/reports/` and `/kaggle/working/fracatlas_reports.zip`.

In [1]:
import os, subprocess, sys
REPO_URL = "https://github.com/MHKhan50/fracatlas-fracture-benchmark.git"
os.chdir("/kaggle/working")
if not os.path.exists("fracatlas-fracture-benchmark"):
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
os.chdir("fracatlas-fracture-benchmark")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
import torch, timm, ultralytics
print("torch", torch.__version__, "| timm", timm.__version__, "| ultralytics", ultralytics.__version__, "| cuda", torch.cuda.is_available())
DATA, RUNS, REPORTS = "/kaggle/working/data", "/kaggle/working/runs", "/kaggle/working/reports"
def fa(*args):
    """Run one CLI step, stream its output into the notebook and stop with the error text if it fails."""
    proc = subprocess.Popen([sys.executable, "-m", "fracatlas_bench.cli", *args, "--data-dir", DATA],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    if proc.wait():
        raise RuntimeError("step failed: " + " ".join(args) + "\n" + "".join(lines[-30:]))

44404ef Detector: read truncated JPEGs with Pillow; skip finished steps and restore them from an earlier run

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
torch 2.11.0+cu130 | timm 1.0.30 | ultralytics 8.4.216 | cuda True


## 1. Data (downloaded from figshare, not redistributed) and the fixed split

In [2]:
fa("download")
fa("split", "--out", REPORTS)

/kaggle/input/datasets/muzammalhussain11/naam-fracatlas/FracAtlas-2 4083 images, 719 fractured
fractured     0    1
split               
test        505  109
train      2354  501
val         505  109
region  hand  hip   leg  multiple  shoulder
split                                      
test     192   27   319        60        16
train    897  125  1486       278        69
val      192   27   319        60        16


## 2. Smoke test: every step on a small subset, one epoch

In [3]:
S = "/kaggle/working/smoke"
fa("probe", "--limit", "120", "--out", S)
for m in ("cnn", "convnext_tiny", "dinov2_small"):
    fa("train", "--model", m, "--limit", "120", "--epochs", "1", "--seeds", "0", "--out", S)
fa("detect", "--limit", "60", "--epochs", "1", "--out", S)
fa("analyse", "--limit", "120", "--out", S, "--reports", S + "/reports")
import shutil; shutil.rmtree(S, ignore_errors=True)

/kaggle/working/fracatlas-fracture-benchmark/src/fracatlas_bench/cli.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(max(2, args.limit // 6))).reset_index(drop=True))
{"model": "dinov2_probe", "seed": 0, "C": 30, "best_val_auroc": 0.78, "features": 384}
/kaggle/working/fracatlas-fracture-benchmark/src/fracatlas_bench/cli.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g

## 3. Full run: frozen probe, three classifiers x three seeds, YOLO

In [4]:
fa("probe", "--out", RUNS)
for m in ("cnn", "convnext_tiny", "dinov2_small"):
    fa("train", "--model", m, "--seeds", "0,1,2", "--out", RUNS)
fa("detect", "--epochs", "60", "--out", RUNS)

{"model": "dinov2_probe", "seed": 0, "C": 30, "best_val_auroc": 0.8932691434281043, "features": 384}
cnn seed=0 epoch=1/25 val_auroc=0.7374
cnn seed=0 epoch=2/25 val_auroc=0.7279
cnn seed=0 epoch=3/25 val_auroc=0.6978
cnn seed=0 epoch=4/25 val_auroc=0.7540
cnn seed=0 epoch=5/25 val_auroc=0.7436
cnn seed=0 epoch=6/25 val_auroc=0.7710
cnn seed=0 epoch=7/25 val_auroc=0.7710
cnn seed=0 epoch=8/25 val_auroc=0.6937
cnn seed=0 epoch=9/25 val_auroc=0.8012
cnn seed=0 epoch=10/25 val_auroc=0.7910
cnn seed=0 epoch=11/25 val_auroc=0.8020
cnn seed=0 epoch=12/25 val_auroc=0.6842
cnn seed=0 epoch=13/25 val_auroc=0.7639
cnn seed=0 epoch=14/25 val_auroc=0.8264
cnn seed=0 epoch=15/25 val_auroc=0.7890
cnn seed=0 epoch=16/25 val_auroc=0.8187
cnn seed=0 epoch=17/25 val_auroc=0.8385
cnn seed=0 epoch=18/25 val_auroc=0.8340
cnn seed=0 epoch=19/25 val_auroc=0.8452
cnn seed=0 epoch=20/25 val_auroc=0.8319
cnn seed=0 epoch=21/25 val_auroc=0.8441
cnn seed=0 epoch=22/25 val_auroc=0.8476
cnn seed=0 epoch=23/25 val_a

## 4. Analysis, released weights and zip

In [5]:
fa("analyse", "--out", RUNS, "--reports", REPORTS)
import glob, shutil, json
os.makedirs(REPORTS + "/runs", exist_ok=True)
for f in glob.glob(RUNS + "/*_logits.npz") + glob.glob(RUNS + "/*_meta.json") + glob.glob(RUNS + "/yolo_*.json") + glob.glob(RUNS + "/yolo_test_scores.npz"):
    shutil.copy(f, REPORTS + "/runs/")
for p in glob.glob(DATA + "/cache_*.npy"):
    os.remove(p)
shutil.make_archive("/kaggle/working/fracatlas_reports", "zip", REPORTS)
print(open(REPORTS + "/analysis.md").read())
print(json.dumps(json.load(open(REPORTS + "/analysis.json"))["released_weights"], indent=1))

Test images: 614 (109 fractured).

| Model | AUROC, mean ± SD | Seed ensemble AUROC [95% CI] | Sens. at 90% spec. | ECE before → after | Refer rate (conf < 0.8) |
|---|---|---|---|---|---|
| Small CNN | 0.791 ± 0.039 | 0.809 [0.763, 0.852] | 0.440 | 0.182 → 0.151 | 0.45 |
| ConvNeXt-Tiny | 0.922 ± 0.006 | 0.928 [0.891, 0.957] | 0.807 | 0.057 → 0.036 | 0.10 |
| DINOv2-S frozen probe | 0.873 ± 0.000 | 0.873 [0.833, 0.909] | 0.642 | 0.139 → 0.129 | 0.36 |
| DINOv2-S fine-tuned | 0.872 ± 0.043 | 0.902 [0.865, 0.934] | 0.743 | 0.092 → 0.085 | 0.25 |

convnext_tiny_minus_cnn: +0.118 [+0.086, +0.151]

convnext_tiny_minus_dinov2_probe: +0.054 [+0.026, +0.082]

convnext_tiny_minus_dinov2_small: +0.026 [+0.005, +0.045]

YOLO test: mAP50 0.321, mAP50-95 0.143, image-level AUROC 0.882

Test images: 614 (109 fractured).

| Model | AUROC, mean ± SD | Seed ensemble AUROC [95% CI] | Sens. at 90% spec. | ECE before → after | Refer rate (conf < 0.8) |
|---|---|---|---|---|---|
| Small CNN | 0.791 ± 0.03